# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [6]:
from pathlib import Path
import time

# ── Chemins (même logique que Sessions 1 et 2) ───────────────────────────────
# "data" = racine du projet ; "../data" = fallback si cwd différent
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
OUTPUT_DIR = DATA_DIR / "output"
VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), (
        f"Fichier manquant : {p.resolve()}\n"
        "Exécutez Spark_DIA3_Session_2.ipynb §2.8 (écriture Parquet) avant ce notebook."
    )

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

print(f"[OK] {VELIB_CONSOLIDE.resolve()}")

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED = 42


[OK] /Users/romain/Desktop/SparkVelib/data/output/disponibilite_consolidee.parquet


In [7]:
import os
import sys
import platform
import subprocess
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# Mac Apple Silicon : Java arm64 + workers PySpark arm64
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"
    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

# Delta Lake : pip install delta-spark (configure les JARs automatiquement)
try:
    spark.stop()
except NameError:
    pass

_builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "4g")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(_builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")


Spark 3.5.0 -- Delta Lake activé
Spark UI : http://localhost:4040


26/07/21 10:29:13 WARN SparkSession: Cannot use io.delta.sql.DeltaSparkSessionExtension to configure session extensions.
java.lang.ClassNotFoundException: io.delta.sql.DeltaSparkSessionExtension
	at java.net.URLClassLoader.findClass(URLClassLoader.java:387)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:418)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:351)
	at java.lang.Class.forName0(Native Method)
	at java.lang.Class.forName(Class.java:351)
	at org.apache.spark.util.SparkClassUtils.classForName(SparkClassUtils.scala:41)
	at org.apache.spark.util.SparkClassUtils.classForName$(SparkClassUtils.scala:36)
	at org.apache.spark.util.Utils$.classForName(Utils.scala:94)
	at org.apache.spark.sql.SparkSession$.$anonfun$applyExtensions$2(SparkSession.scala:1367)
	at org.apache.spark.sql.SparkSession$.$anonfun$applyExtensions$2$adapted(SparkSession.scala:1365)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.fo

In [ ]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [ ]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


In [ ]:
# Première requête : distribution des statuts par arrondissement :
# Nom de 'snapshots', taux d'occuoation moyen, écarrt-type de l'occupation
spark.sql("""
# TODO Requête SQL
""").show(40)


In [ ]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapsohts, nombre de stations par heure
spark.sql("""
# TODO Requête SQL
""").show(24)


---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [ ]:
# Table de jours fériés (2022-2023) -- injectée en broadcast
# Source : légifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2022-01-01",), ("2022-04-18",), ("2022-05-01",), ("2022-05-08",),
    ("2022-05-26",), ("2022-06-06",), ("2022-07-14",), ("2022-08-15",),
    ("2022-11-01",), ("2022-11-11",), ("2022-12-25",),
    ("2023-01-01",), ("2023-04-10",), ("2023-05-01",), ("2023-05-08",),
    ("2023-05-18",), ("2023-05-29",), ("2023-07-14",), ("2023-08-15",),
    ("2023-11-01",), ("2023-11-11",), ("2023-12-25",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2022-2023)")


In [ ]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations trè!s fréquentées,ayant plus de 100 observations (snapshots)
# -> Id de la station, noù de la station, arroondissement, nombre d'observations (snapshots)
df_q1 = spark.sql("""
# TODO Requête SQL
""")

df_q1.show(truncate=False)

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [ ]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
df_q2 = spark.sql("""
# TODO SQL
""")
df_q2.show()


In [ ]:
# Effet de la pluie par arrondissement
df_q2_arr = spark.sql("""
# TODO Requête SQL
""")
df_q2_arr.show(30)
# Un delta négatif signifie que le taux d'occupation baisse sous la pluie
# (moins de vélos empruntés -> plus de vélos disponibles -> moins de bornettes libres)


### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [ ]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire — presque vides à certaines heures, saturées à d'autres ?
df_q3 = spark.sql("""
# TODO Requête SQL
""")
df_q3.show(truncate=False)


In [ ]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# Votre requête ici :
spark.sql("""
    SELECT 'TODO' AS resultat
""").show()


---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [ ]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.

fenetre_station = # Créer un fenêtrage avec l"objet Window

df_avec_lag = (
    df # requête sous forme fonctionnelle
)
df_avec_lag.show(truncate=False)


In [ ]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)

fenetre_heure = # Créer un fenêtrage

df_rank = (
    df # requête sous forme fonctionnelle
)
df_rank.show(48, truncate=False)


In [ ]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

fenetre_cumul = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot précédent (delta instantané)
station_cible = 1042   # à adapter selon vos données

df_delta = (
    df # requête sous forme fonctionnelle
)
df_delta.show(truncate=False)


---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [ ]:
from delta.tables import DeltaTable

# Écriture initiale : toutes les données de 2022
df_2022 = df.filter(F.col("annee") == 2022)

t0 = time.perf_counter()
(
    df_2022
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture 2022 : {time.perf_counter()-t0:.1f} s  --  {df_2022.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")


In [ ]:
# Ajout des données 2023 -- mode "append"
df_2023 = df.filter(F.col("annee") == 2023)

t0 = time.perf_counter()
(
    df_2023 # Enregistrer les données au format DeltaLake, en mode ajout
)
print(f"Ajout 2023 : {time.perf_counter()-t0:.1f} s  --  {df_2023.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)


### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [ ]:
# Lecture de la version 0 (uniquement les données 2022)
df_v0 = (
# TODO Lecture des données
)
print(f"Version 0 (2022 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = # TODO
print(f"Version courante (2022+2023) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [ ]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit, current_timestamp

# On prend 500 snapshots existants et on simule une correction du taux_occupation
df_corrections = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
)

# Quelques nouvelles lignes fictives (snapshots manqués)
df_nouveaux = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(50)
    .withColumn("horodatage",
        F.col("horodatage") + F.expr("INTERVAL 2 YEARS"))
    .withColumn("annee", lit(2024))
    .withColumn("source", lit("nouveau_batch"))
)

df_batch = df_corrections.union(df_nouveaux).drop("source")
print(f"Batch entrant : {df_batch.count()} lignes ({df_corrections.count()} corrections + {df_nouveaux.count()} nouvelles)")


In [ ]:
# MERGE INTO : upsert (update + insert)
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        # Condition de correspondance : même station, même horodatage
        "cible.station_id = source.station_id AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)


In [ ]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
